# **Exercise 2 - Uganda: Loading Real-Time Sensor Data into HydroServer using the Streaming Data Loader**

### **Overview**

In this exercise, we will use the **Kanzenze Hydrological Station in Rwanda** as an example to demonstrate how real-time telemetry data can be loaded into HydroServer.

Kanzenze is an existing hydrological monitoring station that was upgraded and equipped with a modern telemetry system through the **Nile Basin Initiative (NBI) HydroMet project**.

The station provides several hydrological time series through the **Rwanda Water Portal**, including both historical and telemetry observations. For this exercise, we will use the **Stage – Telemetry2** time series, which provides river stage observations from **September 2022 to the present**. The data can be accessed directly through the [Rwanda Water Portal](https://waterportal.rwb.rw/index.php/location_ng_info/259501).

## **Description**

This Jupyter Notebook demonstrates how to create a monitoring site and a datastream in HydroServer to receive observations uploaded using the **Streaming Data Loader**.

The notebook:

1. Connects to HydroServer.
2. Creates a workspace.
3. Creates a monitoring site (Thing).
4. Creates observation method metadata.
5. Creates observed property metadata.
6. Creates a unit for the observed property.
7. Creates a processing level.
8. Creates a datastream to receive observations from the **Streaming Data Loader**.

> **Note:** This notebook does not load the observations. It only creates the HydroServer resources and datastream needed to receive them. The **Streaming Data Loader App** is then configured to automatically load observations into this datastream.

### **Prerequisites**

You must have an account on the HydroServer Playground instance to run this notebook. If you haven't set up your user account yet, go to [HydroServer Playground](https://playground.hydroserver.org) and follow the instructions to create a new user account.

> **Note:** If your Google Colab session disconnects, reconnect to the runtime before continuing. Resources that you have already created in HydroServer will not be lost. Avoid rerunning cells that create resources, as this may result in duplicate resources or errors.

### **References**

Code created by Sara Alonso Vicario, Center for Geospatial Solutions.


## 1. **Getting Started**

---

### **Install hydroserverpy**

For this workshop, we will use Google Colab to run the exercises. Before starting, run the code cell below to install the required version of the hydroserverpy package. The current version of hydroserverpy used for this training is [1.11.3.](https://pypi.org/project/hydroserverpy/)

In [ ]:
!pip install hydroserverpy==1.11.3

### **Import Required Packages**

The following packages and modules are used in this exercise:

- **hydroserverpy** – Connects to HydroServer and allows us to create and manage HydroServer resources programmatically.
- **pandas** – Reads, organizes, and processes historical sensor data.
- **datetime** – Works with dates and times.
- **getpass** – Allows you to enter your HydroServer password securely without displaying it on the screen.

In [ ]:
# Import HydroServer to connect to and manage HydroServer resources
from hydroserverpy import HydroServer
# Import pandas to read, organize, and process the sensor data
import pandas as pd
# Import datetime to work with dates and times
from datetime import datetime
# Import getpass to securely enter your HydroServer password
from getpass import getpass

### **Set the Initial Parameters to Connect to HydroServer**

The first step in interacting with a HydroServer instance is to establish a connection to it. For this example, we will use your username (email) and password because we will create the workspace programmatically.

When you run the code, you will be prompted to enter your password.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**

In [ ]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'svicario@lincolninst.edu' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [ ]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')

### **Get Your Workspace ID**

You need your **Workspace ID** to specify the workspace where the monitoring site and datastream will be created.

In [ ]:
# Enter the name of the workspace you created in Exercise 1
workspace_name = "Uganda Training 2026 - your name"

workspaces = hs.workspaces.list(
    is_associated=True
).fetch_all()

workspace_id = next(
    workspace.uid
    for workspace in workspaces.items
    if workspace.name == workspace_name
)

print("Selected workspace:", workspace_name)
print("Workspace ID:", workspace_id)

## 2. **Create the Monitoring Site**

We are going to load real-time the Akagera River between the Kicukiro and Bugesera districts in Rwanda.

First, we need to create a monitoring site, also referred to as a **Thing** in HydroServer. HydroServer uses a modified version of the [OGC SensorThings API data model](https://www.ogc.org/standards/sensorthings/), where a **Thing** represents a monitoring location where observations are collected.

Once the monitoring site is created, HydroServer automatically assigns it a Universally Unique Identifier (UUID). We can use this UUID to build the URL for the site's landing page in HydroServer.

In [ ]:
# Create a Thing for the Kanzenze Hydrological Station

new_thing = hs.things.create(
    workspace=workspace_id,
    name='Kanzenze Hydrological Station',
    description='Hydrological monitoring station on the Nyabarongo River at Kanzenze, Rwanda.',
    sampling_feature_type='Site',
    sampling_feature_code='259501',
    site_type='Stream',
    elevation_m=1338.0,
    latitude=-2.0613,
    longitude=30.0877,
    admin_area_1='Eastern Province',
    admin_area_2='Bugesera',
    country='RW',
    data_disclaimer='Data provided by the Rwanda Water Resources Board (RWB).',
    is_private=False
)

# Get the ID for the new Thing and print its HydroServer landing page

thing_id = new_thing.uid

print(f'Created new thing with ID: {thing_id}')
print('You can access the new Thing in the HydroServer Data Management App at:')
print(f'{hydroserver_host}/sites/{thing_id}')

## 3. **Create the Datastream**

---

In the following sections, we will create the necessary metadata to load data for a time series of observations recorded at a monitoring site. You can create this metadata using the web user interface of the Data Management App, or you can do it using code, which we are demonstrating here.

HydroServer uses a modified version of the OGC SensorThings API data model for storing time series data and their associated metadata. HydroServer's data model includes the following important entities that we need to create before loading data:

* **Observation Method**: The instrument or method used to measure or create the Observation values.
* **Observed Property**: The variable that is measured (e.g., discharge, water temperature, etc.).
* **Units of Measure**: The units of measure associated with the Observation values (e.g, cubic meters per second).
* **Processing Level**: The degree of processing that has been applied to the Observation values (e.g., "Raw" or "Quality Controlled").
* **Datastream**: A description of the time series that includes all of these attributes.

Once all of these metadata tables have been populated, the time series of data values can be loaded to the **Observations** table in the database.

**NOTE**: To create objects in HydroServer, you will have to pass their required and optional metadata elements. For more information about HydroServer's data model and a data dictionary that describes each of the entities, see HydroServer's documentation at https://www.hydroserver.org.

### **Create an Observation Method (Sensor)**

The OGC SensorThings API data model refers to the method used for creating observations as the "Sensor". In many cases this will be a physical sensor installed at the monitoring site. But, sometimes other methods are used to create observations. We need to create the metadata describing this so potential data users know how the data were created.

**NOTE**: The specific metadata required when creating metadata for a Sensor is dependent upon the "Method Type". For instrument deployments, specific information about the manufacturer and model of the sensor should be specified. For "Derivation" methods, the name and description are required, and a method_code and method_link can be specified if needed.

In [ ]:
real_time_stage_sensor = hs.sensors.create(
    workspace=workspace_id,
    name='Kanzenze Real time Stage Observations',
    description='Real time stage observations recorded at the Kanzenze hydrological station.',
    encoding_type='application/json',
    method_type='Observation',
    method_code='kanzenze-real-time-stage'
)

print("Created observed property:")
print(f"{real_time_stage_sensor.name}: {real_time_stage_sensor.uid}")

### Create an Observed Property

An **Observed Property** defines the variable being measured at a monitoring site. As with the monitoring site (Thing), we need to provide the required and optional metadata that describe the Observed Property.

In this example, we will create an Observed Property for **river stage**, since the CSV file we will upload contains historical river stage observations.

In [ ]:
stage = hs.observedproperties.create(
    workspace=workspace_id,
    name='Stage',
    definition='Stage',
    description='Stage is the height of the water surface at a monitoring location relative to a reference level.',
    observed_property_type='Hydrology',
    code='Stage'
)

print("Created observed property:")
print(f"{stage.name}: {stage.uid}")

### **Create Units of Measure**

Next, we need to add metadata specifying the unit of measurement used for the observations in the CSV file.

In [ ]:
stage_unit = hs.units.create(
    workspace=workspace_id,
    name='Meter',
    symbol='m',
    definition='Unit for water stage',
    unit_type='Length'
)

print("Created unit:")
print(f"{stage_unit.name}: {stage_unit.uid}")

### **Create a Processing Level**

In HydroServer, the Processing Level indicates the degree of processing an observation has been subject to. F

or example, data can be "Raw", which means that they were recorded in the field and nobody has looked at them yet, or they could be "Quality Controlled", which means that a technician has reviewed the data.

We are considering these observations to be raw data with no additional processing, so we need to define a Processing Level that indicates this.

In [ ]:
new_processing_level = hs.processinglevels.create(
    workspace=workspace_id,
    code='Raw',
    definition='Raw Data',
    explanation='Data that have not been processed or quality controlled.'
)

print("Created processing levels:")
print(f"{new_processing_level.code}: {new_processing_level.uid}")

### **Configure the Datastream**

The final step before loading the observations is to create a Datastream. A datastream describes the time series and connects it to the relevant metadata, including where the observations were collected, what variable was measured, which observation method was used, the unit of measurement, and the processing level.

In the following code, we create the datastream by linking the metadata resources created in the previous steps using their unique identifiers (UIDs). We also define additional datastream-specific metadata, such as the data characteristics, time spacing, name, and description.

Once the datastream is created, we can load the observations from the CSV file into it.

**NOTE**: Since these Datastreams are new, they don't contain any Observation values yet. We'll set the ```value_count=0``` and arbitrarily set the ```phenomenon_begin_time``` and ```phenomenon_end_time```. Those will get reset when we load Observation values.


In [ ]:
ds_stage = hs.datastreams.create(
    name=f"{stage.name} - Real-time - {new_thing.name}",
    description=f'Real-time {stage.name.lower()} observations at {new_thing.name}',
    thing=new_thing.uid,
    sensor=real_time_stage_sensor.uid,
    observed_property=stage.uid,
    processing_level=new_processing_level.uid,
    unit=stage_unit.uid,
    observation_type='Field Observation',
    result_type='Timeseries',
    sampled_medium='Surface Water',
    no_data_value=-9999,
    aggregation_statistic='Continuous',
    time_aggregation_interval=0,
    time_aggregation_interval_unit='minutes',
    intended_time_spacing=10,
    intended_time_spacing_unit='minutes',
    status='Ongoing',
    value_count=0,
    phenomenon_begin_time=datetime(year=2026, month=1, day=1),
    is_private=False,
    is_visible=True
)

print("Created datastream:")
print(f"{ds_stage.name}: {ds_stage.uid}")

### What You Have Learned

You now know how to use `hydroserverpy` to:

- Connect to a HydroServer instance.
- Access an existing workspace using its unique identifier (UID).
- Create a monitoring site (Thing).
- Define the required metadata, including the observation method, observed property, unit, and processing level.
- Create a datastream and link it to the corresponding metadata.
- Configure the datastream to receive observations from the **Streaming Data Loader**.